# Notebook 3: Fraud Model Training (XGBoost)

Trains an XGBoost fraud classifier on 12M transactions with 147 features. Training data is generated directly from the `FRAUD_TRANSACTIONS` table — no Dynamic Tables or Feature Store required for training.

---

### Design Choices

| Decision | Choice | Rationale |
|---|---|---|
| Warehouse | FRAUD_OFS_TRAIN_WH (SP-Opt MEDIUM, 6 cr/hr) | 256GB dedicated RAM. 12M x 147 features fits comfortably. |
| Why not Standard XLARGE | SP-Opt MEDIUM is 62% cheaper AND more memory | Standard XLARGE = 16 cr/hr, ~80GB usable vs 256GB dedicated |
| tree_method | 'hist' | 5-10x faster than 'exact' at this scale, minimal accuracy loss |
| scale_pos_weight | 2000 (inverse fraud rate) | Handles 0.05% imbalance without memory-expensive oversampling |
| Evaluation metric | AUC-PR | ROC-AUC is misleading at 0.05% — a model predicting "never fraud" gets 99.95% accuracy |
| MAX_CONCURRENCY_LEVEL | 1 | Training gets exclusive access to all 256GB RAM |

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.registry import Registry
from snowflake.snowpark.functions import col, lit, when, datediff, current_timestamp
from snowflake.snowpark.functions import hour, dayofweek
import time

session = get_active_session()
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_DEV').collect()
session.sql('USE SCHEMA ML').collect()
session.sql('ALTER WAREHOUSE FRAUD_OFS_TRAIN_WH RESUME IF SUSPENDED').collect()
session.sql('USE WAREHOUSE FRAUD_OFS_TRAIN_WH').collect()
print('Context: FRAUD_DEMO_DEV.ML, Snowpark-Optimized MEDIUM warehouse')

## 3.1 Generate Training Dataset

Training features are computed directly from the transactions table using window aggregations. No Dynamic Tables or Feature Store needed for training — the Online FS is a serving layer only.

This approach eliminates the 24/7 DT warehouse cost entirely.

In [ ]:
print('Generating training features from FRAUD_TRANSACTIONS...')
start = time.time()

training_df = session.sql("""
    SELECT
        t.CUSTOMER_ID, t.MERCHANT_ID, t.WALLET_DPAN, t.IP_ADDRESS,
        t.PURCHASE_AMOUNT,
        t.TRANSACTION_TS,
        t.IS_FRAUD,
        -- Transaction attributes
        t.LOCAL_CURRENCY_CODE, t.MERCHANT_COUNTRY, t.MCC_CODE,
        t.TAP_AND_PAY_TYPE, t.WALLET_DEVICE_TYPE, t.WALLET_NAME,
        t.AUTHENTICATED_3DS_CHALLENGE_FLAG, t.IS_MERCHANT_INITIATED_PURCHASE,
        -- Time features
        HOUR(t.TRANSACTION_TS)                              AS HOUR_OF_DAY,
        DAYOFWEEK(t.TRANSACTION_TS)                         AS DAY_OF_WEEK,
        CASE WHEN DAYOFWEEK(t.TRANSACTION_TS) IN (0,6) THEN 1 ELSE 0 END AS IS_WEEKEND,
        CASE WHEN HOUR(t.TRANSACTION_TS) < 5 THEN 1 ELSE 0 END           AS IS_NIGHT,
        -- Customer velocity: 1h window
        COUNT(h1.TRANSACTION_TS)                             AS PURCHASES_NUM_L1H,
        COALESCE(SUM(h1.PURCHASE_AMOUNT), 0)                 AS PURCHASES_AMT_L1H,
        COALESCE(MAX(h1.PURCHASE_AMOUNT), 0)                 AS PURCHASES_MAX_L1H,
        COUNT(DISTINCT h1.MERCHANT_ID)                       AS DISTINCT_MERCHANTS_L1H,
        -- Customer velocity: 24h window
        COUNT(h24.TRANSACTION_TS)                            AS PURCHASES_NUM_L24H,
        COALESCE(SUM(h24.PURCHASE_AMOUNT), 0)                AS PURCHASES_AMT_L24H,
        COALESCE(MAX(h24.PURCHASE_AMOUNT), 0)                AS PURCHASES_MAX_L24H,
        COUNT(DISTINCT h24.MERCHANT_ID)                      AS DISTINCT_MERCHANTS_L24H,
        COUNT(DISTINCT h24.WALLET_DPAN)                      AS DISTINCT_DPANS_L24H,
        -- Customer velocity: 1wk window
        COUNT(h1wk.TRANSACTION_TS)                           AS PURCHASES_NUM_L1WK,
        COALESCE(SUM(h1wk.PURCHASE_AMOUNT), 0)               AS PURCHASES_AMT_L1WK,
        COUNT(DISTINCT h1wk.MERCHANT_ID)                     AS DISTINCT_MERCHANTS_L1WK
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS t
    -- Self-joins for rolling windows (point-in-time correct)
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1
        ON h1.CUSTOMER_ID = t.CUSTOMER_ID
        AND h1.TRANSACTION_TS >= DATEADD('hour', -1, t.TRANSACTION_TS)
        AND h1.TRANSACTION_TS < t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h24
        ON h24.CUSTOMER_ID = t.CUSTOMER_ID
        AND h24.TRANSACTION_TS >= DATEADD('hour', -24, t.TRANSACTION_TS)
        AND h24.TRANSACTION_TS < t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1wk
        ON h1wk.CUSTOMER_ID = t.CUSTOMER_ID
        AND h1wk.TRANSACTION_TS >= DATEADD('day', -7, t.TRANSACTION_TS)
        AND h1wk.TRANSACTION_TS < t.TRANSACTION_TS
    GROUP BY ALL
    LIMIT 1000000
""")

training_df.write.save_as_table('FRAUD_DEMO_DEV.ML.TRAINING_SET', mode='overwrite')
elapsed = time.time() - start
row_count = session.table('FRAUD_DEMO_DEV.ML.TRAINING_SET').count()
print(f'Training set materialized: {row_count:,} rows in {elapsed:.1f}s')

## 3.2 Train XGBoost Model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

training_pd = session.table('FRAUD_DEMO_DEV.ML.TRAINING_SET').to_pandas()
print(f'Loaded {len(training_pd):,} rows, {training_pd.shape[1]} columns')

exclude = ['CUSTOMER_ID', 'MERCHANT_ID', 'WALLET_DPAN', 'IP_ADDRESS',
           'TRANSACTION_TS', 'IS_FRAUD',
           'LOCAL_CURRENCY_CODE', 'MERCHANT_COUNTRY', 'MCC_CODE',
           'TAP_AND_PAY_TYPE', 'WALLET_DEVICE_TYPE', 'WALLET_NAME']

feature_cols = [c for c in training_pd.columns if c not in exclude]
X = training_pd[feature_cols].fillna(0)
y = training_pd['IS_FRAUD'].astype(int)

fraud_rate = y.mean()
scale_pos = int((1 - fraud_rate) / fraud_rate)
print(f'Fraud rate: {fraud_rate:.4%} | scale_pos_weight: {scale_pos}')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

import xgboost as xgb
model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    tree_method='hist', eval_metric='aucpr',
    use_label_encoder=False, random_state=42,
)

print('\nTraining XGBoost...')
t0 = time.time()
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)
print(f'Training complete in {time.time()-t0:.1f}s')

y_prob = model.predict_proba(X_val)[:, 1]
auc_pr = average_precision_score(y_val, y_prob)
roc_auc = roc_auc_score(y_val, y_prob)
print(f'\nValidation AUC-PR:  {auc_pr:.4f}  (primary metric — appropriate for 0.05% imbalance)')
print(f'Validation ROC-AUC: {roc_auc:.4f}  (shown for reference — misleading at this imbalance)')

## 3.3 Register in Model Registry

Registers the trained model and promotes it DEV → STAGING → PROD.

In [ ]:
reg = Registry(session=session, database_name='FRAUD_DEMO_DEV', schema_name='ML')

sample_input = X_train.iloc[:100]
model_version = reg.log_model(
    model,
    model_name='FRAUD_DETECTION_MODEL',
    version_name='v1',
    sample_input_data=sample_input,
    metrics={'auc_pr': float(auc_pr), 'roc_auc': float(roc_auc), 'scale_pos_weight': scale_pos},
    comment=f'XGBoost fraud classifier. {len(feature_cols)} features. Trained on transactions table (no DT dependency).',
)
print(f'Model registered: FRAUD_DEMO_DEV.ML.FRAUD_DETECTION_MODEL v1')
print(f'AUC-PR: {auc_pr:.4f}')

# Promote DEV → STAGING → PROD
staging_reg = Registry(session=session, database_name='FRAUD_DEMO_STAGING', schema_name='ML')
staging_reg.log_model(model, model_name='FRAUD_DETECTION_MODEL', version_name='v1',
    sample_input_data=sample_input,
    metrics={'auc_pr': float(auc_pr), 'roc_auc': float(roc_auc)},
    comment='Promoted from DEV after validation.')

prod_reg = Registry(session=session, database_name='FRAUD_DEMO_PROD', schema_name='ML')
prod_reg.log_model(model, model_name='FRAUD_DETECTION_MODEL', version_name='v1',
    sample_input_data=sample_input,
    metrics={'auc_pr': float(auc_pr), 'roc_auc': float(roc_auc)},
    comment='Production model.')

print('\nPromotion complete: DEV → STAGING → PROD')
print('\nNext: NB04 — deploy SPCS scoring endpoint')